# 8Chess Custom NNUE: Leela Chess Zero (Lc0) GPU Trainer
This notebook trains a custom, high-performance NNUE network for **8Chess** (Reckless fork) on Kaggle GPU from Leela Chess Zero (Lc0) self-play games.

- **Architecture:** 7,680 piece features + 66,864 threat features -> 768 FT -> Clipped-Pair Squaring -> L1 16 (8 buckets) -> L2 32 -> L3 1
- **File Format:** Exact little-endian struct serialization (63,266,880 bytes)
- **Target Hardware:** T4 / P100 GPU on Kaggle


In [ ]:
!pip install python-chess -q
import os
import sys
import glob
import time
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import chess
import chess.pgn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1. 8Chess NNUE Architecture Constants & Feature Tables


In [ ]:
# --- Architecture Definitions & Feature Extractors ---
import os
os.makedirs("eightchess_nnue", exist_ok=True)
with open("eightchess_nnue/__init__.py", "w") as f:
    f.write("")

with open("eightchess_nnue/constants.py", "w") as f:
    f.write('"""Architecture constants for the 8Chess (Reckless-lineage) NNUE.\n\nThese MUST stay in sync with `src/nnue.rs` in the 8Chess engine.\nSee docs/nnue_format.md for the full binary layout specification.\n"""\n\n# Feature transformer\nL1_SIZE = 768\nL2_SIZE = 16\nL3_SIZE = 32\n\nINPUT_BUCKETS = 10\nOUTPUT_BUCKETS = 8\n\n# Threat feature space (66864 = total sized from piece interaction tables)\nTHREAT_FEATURES = 66864\nPIECE_FEATURES = INPUT_BUCKETS * 768  # 7680\n\n# Quantization (must match src/nnue.rs)\nFT_QUANT = 255\nL1_QUANT = 64\nFT_SHIFT = 9  # scalar inference uses (left * right) >> FT_SHIFT\n\nDEQUANT_MULTIPLIER = float(1 << FT_SHIFT) / float(FT_QUANT * FT_QUANT * L1_QUANT)\n\nNETWORK_SCALE = 380\n\n# Input bucket layout indexed by square (a1=0 .. h8=63)\nINPUT_BUCKETS_LAYOUT = [\n    0, 1, 2, 3, 3, 2, 1, 0,\n    4, 5, 6, 7, 7, 6, 5, 4,\n    8, 8, 8, 8, 8, 8, 8, 8,\n    9, 9, 9, 9, 9, 9, 9, 9,\n    9, 9, 9, 9, 9, 9, 9, 9,\n    9, 9, 9, 9, 9, 9, 9, 9,\n    9, 9, 9, 9, 9, 9, 9, 9,\n    9, 9, 9, 9, 9, 9, 9, 9,\n]\n\n# Output bucket layout indexed by piece count\nOUTPUT_BUCKETS_LAYOUT = [\n    0, 0, 0, 0, 0, 0, 0, 0, 0,\n    1, 1, 1, 1,\n    2, 2, 2, 2,\n    3, 3, 3,\n    4, 4, 4,\n    5, 5, 5,\n    6, 6, 6,\n    7, 7, 7, 7,\n]\n\n# Exact serialized size of the Parameters struct (bytes)\nNETWORK_SIZE = 63_266_880\n\n\ndef output_bucket(piece_count: int) -> int:\n    return OUTPUT_BUCKETS_LAYOUT[piece_count]\n\n\n# ---------------------------------------------------------------------------\n# Threat feature indexing tables (port of src/nnue/accumulator/threats/\n# threat_index.rs). Built once on import.\n# ---------------------------------------------------------------------------\n\n# Rows/columns: PieceType (P,N,B,R,Q,K). -1 == excluded interaction.\nPIECE_INTERACTION_MAP = [\n    [0, 1, -1, 2, -1, -1],\n    [0, 1, 2, 3, 4, -1],\n    [0, 1, 2, 3, -1, -1],\n    [0, 1, 2, 3, -1, -1],\n    [0, 1, 2, 3, 4, -1],\n    [0, 1, 2, 3, -1, -1],\n]\n\nPIECE_TARGET_COUNT = [6, 10, 8, 8, 10, 8]\n\n\ndef _build_attack_tables():\n    """Attack sets (as 64-bit masks) for offsets 0..63, occupancy 0.\n\n    Mirrors the generated tables in the engine for the purpose of\n    (a) counting target squares per (piece, square) and\n    (b) computing the "number of attacked squares below `to`" indices.\n    """\n    knight = [0] * 64\n    king = [0] * 64\n    pawn = [[0] * 64 for _ in range(2)]  # [color][square]\n    ray_diag = [[0] * 64 for _ in range(4)]\n    ray_orth = [[0] * 64 for _ in range(4)]\n\n    knight_deltas = [(1, 2), (2, 1), (2, -1), (1, -2), (-1, -2), (-2, -1), (-2, 1), (-1, 2)]\n    king_deltas = [(1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1)]\n    diag_deltas = [(1, 1), (-1, 1), (-1, -1), (1, -1)]\n    orth_deltas = [(1, 0), (-1, 0), (0, 1), (0, -1)]\n\n    def add_deltas(deltas, single_step):\n        table = [0] * 64\n        for sq in range(64):\n            f, r = sq & 7, sq >> 3\n            bb = 0\n            for df, dr in deltas:\n                nf, nr = f, r\n                while True:\n                    nf, nr = nf + df, nr + dr\n                    if not (0 <= nf < 8 and 0 <= nr < 8):\n                        break\n                    bb |= 1 << (nr * 8 + nf)\n                    if single_step:\n                        break\n            table[sq] = bb\n        return table\n\n    knight = add_deltas(knight_deltas, True)\n    king = add_deltas(king_deltas, True)\n    ray_diag = add_deltas(diag_deltas, False)\n    ray_orth = add_deltas(orth_deltas, False)\n\n    for sq in range(64):\n        f, r = sq & 7, sq >> 3\n        # white pawn attacks (towards higher ranks)\n        w = 0\n        for df in (-1, 1):\n            nf, nr = f + df, r + 1\n            if 0 <= nf < 8 and 0 <= nr < 8:\n                w |= 1 << (nr * 8 + nf)\n        pawn[0][sq] = w\n        b = 0\n        for df in (-1, 1):\n            nf, nr = f + df, r - 1\n            if 0 <= nf < 8 and 0 <= nr < 8:\n                b |= 1 << (nr * 8 + nf)\n        pawn[1][sq] = b\n\n    return {\n        "knight": knight,\n        "king": king,\n        "pawn": pawn,\n        "ray_diag": ray_diag,\n        "ray_orth": ray_orth,\n    }\n\n\n_ATTACKS = _build_attack_tables()\n\n\ndef attacks_empty(piece: int, square: int) -> int:\n    """Attack mask with empty occupancy. piece: 0..11 (type*2 + color)."""\n    ptype = piece >> 1\n    color = piece & 1\n    if ptype == 0:\n        return _ATTACKS["pawn"][color][square]\n    if ptype == 1:\n        return _ATTACKS["knight"][square]\n    if ptype == 2:\n        return _ATTACKS["ray_diag"][square]\n    if ptype == 3:\n        return _ATTACKS["ray_orth"][square]\n    if ptype == 4:\n        return _ATTACKS["ray_diag"][square] | _ATTACKS["ray_orth"][square]\n    if ptype == 5:\n        return _ATTACKS["king"][square]\n    return 0\n\n\ndef _build_threat_lookups():\n    piece_offset = [0] * 12\n    offset_table = [0] * 12\n\n    offset = 0\n    for color in (0, 1):\n        for ptype in range(6):\n            piece = ptype * 2 + color\n            count = 0\n            for sq in range(64):\n                if ptype != 0 or 8 <= sq < 56:\n                    count += bin(attacks_empty(piece, sq)).count("1")\n            piece_offset[piece] = count\n            offset_table[piece] = offset\n            offset += PIECE_TARGET_COUNT[ptype] * count\n\n    total = offset\n    assert total == THREAT_FEATURES, (total, THREAT_FEATURES)\n\n    # pair base lookup + exclusion flags\n    pair_base = [[0] * 12 for _ in range(12)]\n    pair_excluded = [[False] * 12 for _ in range(12)]\n    pair_semi = [[False] * 12 for _ in range(12)]\n\n    for attacking in range(12):\n        atk_type, atk_color = attacking >> 1, attacking & 1\n        for attacked in range(12):\n            def_type, def_color = attacked >> 1, attacked & 1\n            m = PIECE_INTERACTION_MAP[atk_type][def_type]\n            base = (\n                offset_table[attacking]\n                + (def_color * (PIECE_TARGET_COUNT[atk_type] // 2) + m) * piece_offset[attacking]\n            )\n            enemy = atk_color != def_color\n            semi = atk_type == def_type and (enemy or atk_type != 0)\n            pair_base[attacking][attacked] = base\n            pair_excluded[attacking][attacked] = m < 0\n            pair_semi[attacking][attacked] = semi\n\n    piece_offsets = [None] * 12\n    attack_index = [None] * 12\n    for piece in range(12):\n        piece_offsets[piece] = [0] * 64\n        count = 0\n        for sq in range(64):\n            piece_offsets[piece][sq] = count\n            if (piece >> 1) != 0 or 8 <= sq < 56:\n                count += bin(attacks_empty(piece, sq)).count("1")\n        assert count == piece_offset[piece]\n\n        table = [0] * 4096\n        for frm in range(64):\n            atk = attacks_empty(piece, frm)\n            for to in range(64):\n                table[frm * 64 + to] = bin(atk & ((1 << to) - 1)).count("1")\n        attack_index[piece] = table\n\n    return {\n        "pair_base": pair_base,\n        "pair_excluded": pair_excluded,\n        "pair_semi": pair_semi,\n        "piece_offsets": piece_offsets,\n        "attack_index": attack_index,\n    }\n\n\nTHREAT_LOOKUPS = _build_threat_lookups()\n')

with open("eightchess_nnue/features.py", "w") as f:
    f.write('"""Exact feature extraction for the 8Chess NNUE (port of the Rust code).\n\nProvides piece (PSQT) and threat feature indices for a python-chess board,\nmatching `src/nnue/accumulator/psq.rs` and\n`src/nnue/accumulator/threats/threat_index.rs` bit for bit.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Iterable, Optional\n\nimport chess\n\nfrom .constants import (\n    INPUT_BUCKETS_LAYOUT,\n    PIECE_FEATURES,\n    THREAT_FEATURES,\n    THREAT_LOOKUPS,\n    attacks_empty,\n)\n\n_PIECE_CODE = {\n    (chess.PAWN, chess.WHITE): 0,\n    (chess.PAWN, chess.BLACK): 1,\n    (chess.KNIGHT, chess.WHITE): 2,\n    (chess.KNIGHT, chess.BLACK): 3,\n    (chess.BISHOP, chess.WHITE): 4,\n    (chess.BISHOP, chess.BLACK): 5,\n    (chess.ROOK, chess.WHITE): 6,\n    (chess.ROOK, chess.BLACK): 7,\n    (chess.QUEEN, chess.WHITE): 8,\n    (chess.QUEEN, chess.BLACK): 9,\n    (chess.KING, chess.WHITE): 10,\n    (chess.KING, chess.BLACK): 11,\n}\n\n_PTYPE_CODE = {\n    chess.PAWN: 0,\n    chess.KNIGHT: 1,\n    chess.BISHOP: 2,\n    chess.ROOK: 3,\n    chess.QUEEN: 4,\n    chess.KING: 5,\n}\n\n_KNIGHT_DELTAS = [(1, 2), (2, 1), (2, -1), (1, -2), (-1, -2), (-2, -1), (-2, 1), (-1, 2)]\n_KING_DELTAS = [(1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1)]\n_DIAG_DELTAS = [(1, 1), (-1, 1), (-1, -1), (1, -1)]\n_ORTH_DELTAS = [(1, 0), (-1, 0), (0, 1), (0, -1)]\n\n\ndef _ray_attacks(sq: int, deltas, occ: int) -> int:\n    bb = 0\n    f, r = sq & 7, sq >> 3\n    for df, dr in deltas:\n        nf, nr = f, r\n        while True:\n            nf, nr = nf + df, nr + dr\n            if not (0 <= nf < 8 and 0 <= nr < 8):\n                break\n            s = nr * 8 + nf\n            bb |= 1 << s\n            if (occ >> s) & 1:\n                break\n    return bb\n\n\ndef attacks_with_occupancy(piece: int, square: int, occ: int) -> int:\n    ptype = piece >> 1\n    if ptype in (2, 4):\n        bb = _ray_attacks(square, _DIAG_DELTAS, occ)\n        if ptype == 4:\n            bb |= _ray_attacks(square, _ORTH_DELTAS, occ)\n        return bb\n    if ptype == 3:\n        return _ray_attacks(square, _ORTH_DELTAS, occ)\n    return attacks_empty(piece, square)\n\n\ndef iter_bits(bb: int) -> Iterable[int]:\n    while bb:\n        lsb = bb & -bb\n        yield lsb.bit_length() - 1\n        bb ^= lsb\n\n\ndef piece_feature_indices(board: chess.Board) -> list[list[int]]:\n    """Returns [pov0_features, pov1_features] where pov matches side-to-move order.\n\n    pov0 == White, pov1 == Black: the engine refreshes accumulators for both\n    colors; the activation then selects by side to move.\n    """\n    result: list[list[int]] = []\n    for pov in (chess.WHITE, chess.BLACK):\n        king = board.king(pov)\n        assert king is not None\n        kingside = (king & 7) >= 4\n        flip = (7 if kingside else 0) ^ (56 if pov == chess.BLACK else 0)\n        bucket = INPUT_BUCKETS_LAYOUT[king ^ flip]\n\n        feats = []\n        for sq in chess.scan_reversed(board.occupied):\n            piece = board.piece_at(sq)\n            assert piece is not None\n            color = 0 if piece.color == chess.WHITE else 1\n            ptype = _PTYPE_CODE[piece.piece_type]\n            pov_code = 0 if pov == chess.WHITE else 1\n            idx = bucket * 768 + 384 * (color != pov_code) + 64 * ptype + (sq ^ flip)\n            assert 0 <= idx < PIECE_FEATURES\n            feats.append(idx)\n        result.append(feats)\n    return result\n\n\ndef threat_index(piece: int, frm: int, attacked: int, to: int, mirrored: bool, pov: int) -> Optional[int]:\n    """Port of `threat_index` in Rust. Returns None when the pair/square\n    ordering is excluded (symmetric interactions are stored once)."""\n    if pov == 1:\n        frm ^= 56\n        to ^= 56\n    if mirrored:\n        frm ^= 7\n        to ^= 7\n\n    attacking = piece ^ pov\n    attacked_p = attacked ^ pov\n\n    look = THREAT_LOOKUPS\n    if look["pair_excluded"][attacking][attacked_p]:\n        return None\n    if look["pair_semi"][attacking][attacked_p] and frm < to:\n        return None\n\n    base = look["pair_base"][attacking][attacked_p]\n    idx = base + look["piece_offsets"][attacking][frm] + look["attack_index"][attacking][frm * 64 + to]\n    assert 0 <= idx < THREAT_FEATURES, idx\n    return idx\n\n\ndef threat_feature_indices(board: chess.Board) -> list[list[int]]:\n    """Returns [pov0_features, pov1_features] with pov = White/Black."""\n    occ = board.occupied\n    result: list[list[int]] = []\n\n    for pov in (chess.WHITE, chess.BLACK):\n        king = board.king(pov)\n        assert king is not None\n        mirrored = (king & 7) >= 4\n        pov_code = 0 if pov == chess.WHITE else 1\n\n        feats = []\n        for sq in chess.scan_reversed(occ):\n            piece_obj = board.piece_at(sq)\n            assert piece_obj is not None\n            piece = _PIECE_CODE[(piece_obj.piece_type, piece_obj.color)]\n            targets = attacks_with_occupancy(piece, sq, occ) & occ\n            for to in iter_bits(targets):\n                target = board.piece_at(to)\n                assert target is not None\n                attacked = _PIECE_CODE[(target.piece_type, target.color)]\n                idx = threat_index(piece, sq, attacked, to, mirrored, pov_code)\n                if idx is not None:\n                    feats.append(idx)\n        result.append(feats)\n    return result\n\n\ndef full_feature_indices(board: chess.Board):\n    """(piece_features, threat_features), each [pov0, pov1]."""\n    return piece_feature_indices(board), threat_feature_indices(board)\n')

with open("eightchess_nnue/format.py", "w") as f:
    f.write('"""Binary format read/write for the 8Chess NNUE `Parameters` image.\n\nThe engine deserializes the network with a raw `transmute` of the file\ncontents into a `#[repr(C)]` struct (see src/nnue.rs). This module is the\ncanonical specification of that layout: every offset is asserted, and\nexported files are guaranteed byte-compatible with `EvalFile`.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .constants import (\n    INPUT_BUCKETS,\n    L1_SIZE,\n    L2_SIZE,\n    L3_SIZE,\n    NETWORK_SIZE,\n    OUTPUT_BUCKETS,\n    PIECE_FEATURES,\n    THREAT_FEATURES,\n)\n\n# (name, dtype, shape, byte offset) -- offsets must match src/nnue.rs exactly.\n_LAYOUT = [\n    ("ft_threat_weights", np.int8, (THREAT_FEATURES, L1_SIZE), 0),\n    ("ft_piece_weights", np.int16, (PIECE_FEATURES, L1_SIZE), 51_351_552),\n    ("ft_biases", np.int16, (L1_SIZE,), 63_148_032),\n    ("l1_weights", np.int8, (OUTPUT_BUCKETS, L2_SIZE * L1_SIZE), 63_149_568),\n    ("l1_biases", np.float32, (OUTPUT_BUCKETS, L2_SIZE), 63_247_872),\n    ("l2_weights", np.float32, (OUTPUT_BUCKETS, L2_SIZE, L3_SIZE), 63_248_384),\n    ("l2_biases", np.float32, (OUTPUT_BUCKETS, L3_SIZE), 63_264_768),\n    ("l3_weights", np.float32, (OUTPUT_BUCKETS, L3_SIZE), 63_265_792),\n    ("l3_biases", np.float32, (OUTPUT_BUCKETS,), 63_266_816),\n]\n\n\ndef validate_layout() -> None:\n    end = 0\n    for name, dtype, shape, offset in _LAYOUT:\n        assert offset == end or name == _LAYOUT[0][0], f"gap before {name}: {end} != {offset}"\n        size = int(np.prod(shape)) * np.dtype(dtype).itemsize\n        # fields are 64-byte aligned; every field except the last is a multiple\n        assert offset % 64 == 0, name\n        end = offset + size\n    assert end <= NETWORK_SIZE\n    assert NETWORK_SIZE - end < 64  # trailing padding only\n    assert NETWORK_SIZE == 63_266_880\n\n\nvalidate_layout()\n\n\ndef read_network(path: str) -> dict[str, np.ndarray]:\n    data = np.fromfile(path, dtype=np.uint8)\n    if data.size != NETWORK_SIZE:\n        raise ValueError(f"invalid network size: expected {NETWORK_SIZE}, got {data.size}")\n\n    net = {}\n    for name, dtype, shape, offset in _LAYOUT:\n        size = int(np.prod(shape)) * np.dtype(dtype).itemsize\n        raw = data[offset : offset + size]\n        net[name] = raw.view(dtype).reshape(shape).copy()\n    return net\n\n\ndef build_network(net: dict[str, np.ndarray]) -> bytes:\n    buf = np.zeros(NETWORK_SIZE, dtype=np.uint8)\n\n    for name, dtype, shape, offset in _LAYOUT:\n        arr = np.ascontiguousarray(net[name], dtype=dtype)\n        if arr.shape != shape:\n            raise ValueError(f"{name}: expected shape {shape}, got {arr.shape}")\n        size = int(np.prod(shape)) * np.dtype(dtype).itemsize\n        buf[offset : offset + size] = arr.view(np.uint8).reshape(-1)\n\n    return buf.tobytes()\n\n\ndef write_network(net: dict[str, np.ndarray], path: str) -> None:\n    with open(path, "wb") as f:\n        f.write(build_network(net))\n\n\ndef zero_network() -> dict[str, np.ndarray]:\n    return {\n        name: np.zeros(shape, dtype=dtype)\n        for name, dtype, shape, _ in _LAYOUT\n    }\n')

with open("eightchess_nnue/quantized.py", "w") as f:
    f.write('"""Bit-accurate replica of the engine\'s SIMD inference for AVX2 (and the\nscalar path, which is semantically equal for the activation step).\n\nSemantics captured from src/nnue/forward/vectorized.rs + src/nnue/simd/avx2.rs:\n  - accumulator sums are wrapping i16 arithmetic (mod 2^16)\n  - activation  : s = wrap16(pst + threat); lhs = clamp(s,0,255);\n                  rhs = min(s,255); byte = sat_u8(floor(lhs*rhs / 512))\n  - L1 matmul   : maddubs pairwise sums saturate to i16, chunk-pair sums wrap,\n                  then widen to i32 (madd) and accumulate\n  - float head  : f32 fma in numpy (non-fused mul+add; <=1cp deviation possible)\n"""\n\nfrom __future__ import annotations\n\nimport chess\nimport numpy as np\n\nfrom .constants import (\n    DEQUANT_MULTIPLIER,\n    FT_QUANT,\n    FT_SHIFT,\n    L1_SIZE,\n    L2_SIZE,\n    L3_SIZE,\n    NETWORK_SCALE,\n    output_bucket,\n)\nfrom .features import piece_feature_indices, threat_feature_indices\n\n_CHUNKS = L1_SIZE // 4  # 192 four-byte chunks\n\n\ndef _wrap16(x: np.ndarray) -> np.ndarray:\n    return (x & 0xFFFF) - ((x & 0x8000) << 1)\n\n\ndef _sat16(x: np.ndarray) -> np.ndarray:\n    return np.clip(x, -32768, 32767)\n\n\ndef _accumulate(weights: np.ndarray, indices: list[int], base: np.ndarray, dtype: np.dtype) -> np.ndarray:\n    """wrapping i16 accumulator: base + sum(weights[idx]) mod 2^16."""\n    if not indices:\n        return base\n    idx = np.asarray(indices, dtype=np.int64)\n    summed = weights[idx].astype(np.int64).sum(axis=0)\n    return _wrap16(base.astype(np.int64) + summed).astype(dtype)\n\n\ndef evaluate(net: dict[str, np.ndarray], board: chess.Board) -> int:\n    stm = 0 if board.turn == chess.WHITE else 1\n\n    pst_feats = piece_feature_indices(board)\n    thr_feats = threat_feature_indices(board)\n\n    ft_piece = net["ft_piece_weights"]\n    ft_threat = net["ft_threat_weights"]\n    ft_biases = net["ft_biases"]\n\n    acc_pst = [None, None]\n    acc_thr = [None, None]\n    for pov in (0, 1):\n        acc_pst[pov] = _accumulate(ft_piece, pst_feats[pov], ft_biases, np.int32)\n        acc_thr[pov] = _accumulate(ft_threat, thr_feats[pov], np.zeros(L1_SIZE, dtype=np.int64), np.int32)\n\n    # Activation (per flip ordering: flip 0 uses stm pov, flip 1 the opponent).\n    ft_out = np.zeros(L1_SIZE, dtype=np.int64)\n    for flip in (0, 1):\n        pov = stm ^ flip\n        s = _wrap16(acc_pst[pov].astype(np.int64) + acc_thr[pov].astype(np.int64))\n        lhs = np.clip(s[0 : L1_SIZE // 2], 0, FT_QUANT)\n        rhs = np.minimum(s[L1_SIZE // 2 :], FT_QUANT)\n        prod = (lhs * rhs) >> FT_SHIFT  # floor division for arithmetic shift\n        ft_out[flip * (L1_SIZE // 2) : (flip + 1) * (L1_SIZE // 2)] = np.clip(prod, 0, 255)\n\n    ft_bytes = ft_out.astype(np.uint8)\n    bucket = output_bucket(int(board.occupied.bit_count()))\n\n    # L1: maddubs/dpbusd semantics over 4-byte chunks. The engine pairs\n    # *nonzero* chunks in nnz order (ascending), with the odd tail handled\n    # by a single dpbusd.\n    l1_w = net["l1_weights"][bucket].reshape(_CHUNKS, L2_SIZE, 4).astype(np.int64)\n    bytes_c = ft_bytes.reshape(_CHUNKS, 4)\n\n    nnz = np.nonzero(bytes_c.any(axis=1))[0]\n    pre = np.zeros(L2_SIZE, dtype=np.int64)\n    if nnz.size:\n        prod = bytes_c[nnz].astype(np.int64)[:, None, :] * l1_w[nnz]  # (k, L2, 4)\n        pairwise = _sat16(prod[:, :, 0::2] + prod[:, :, 1::2])  # (k, L2, 2)\n\n        k = nnz.size\n        pairs = k - (k % 2)\n        if pairs:\n            q = _wrap16(pairwise[0:pairs:2] + pairwise[1:pairs:2])  # (pairs/2, L2, 2)\n            pre += q.sum(axis=(0, 2))\n        if k % 2:\n            pre += pairwise[k - 1].sum(axis=1)\n\n    l1_b = net["l1_biases"][bucket].astype(np.float32)\n    l1_out = np.clip(pre.astype(np.float32) * np.float32(DEQUANT_MULTIPLIER) + l1_b, 0.0, 1.0)\n\n    l2_w = net["l2_weights"][bucket].astype(np.float32)\n    l2_b = net["l2_biases"][bucket].astype(np.float32)\n    l2_out = l2_b.copy()\n    for i in range(L2_SIZE):\n        l2_out = l2_w[i] * l1_out[i] + l2_out\n    l2_out = np.clip(l2_out, 0.0, 1.0)\n\n    l3_w = net["l3_weights"][bucket].astype(np.float32)\n    l3_b = np.float32(net["l3_biases"][bucket])\n\n    # AVX2 horizontal_sum: LANES=2 vectors of 8; sum vectors, then tree reduce.\n    v = l3_w[0:8] * l2_out[0:8] + l3_w[16:24] * l2_out[16:24]\n    w = l3_w[8:16] * l2_out[8:16] + l3_w[24:32] * l2_out[24:32]\n    vec = v + w\n    lo4 = vec[0:4] + vec[4:8]\n    two = lo4[0:2] + lo4[2:4]\n    total = two[0] + two[1]\n    l3_out = np.float32(total + l3_b)\n    return int(np.float32(l3_out * np.float32(NETWORK_SCALE)))\n\n\ndef evaluate_fen(net: dict[str, np.ndarray], fen: str) -> int:\n    return evaluate(net, chess.Board(fen))\n')

with open("eightchess_nnue/encoder.py", "w") as f:
    f.write('"""Position encoding for training: features -> ragged index streams.\n\nProduces the four index streams consumed by `model.EightChessNNUE`:\n\n    piece_stm / threat_stm : features of the side-to-move accumulator\n    piece_opp / threat_opp : features of the opponent accumulator\n\nplus the output bucket and the result in side-to-move perspective.\nEncoding is CPU-bound python-chess work; `encode_fens` parallelizes it.\n"""\n\nfrom __future__ import annotations\n\nimport multiprocessing as mp\nfrom functools import partial\n\nimport chess\nimport numpy as np\n\nfrom .constants import output_bucket\nfrom .features import piece_feature_indices, threat_feature_indices\n\n\ndef encode_board(board: chess.Board) -> tuple[list[int], list[int], list[int], list[int], int, int]:\n    """Returns (pieces_stm, threats_stm, pieces_opp, threats_opp, bucket, stm)."""\n    stm = 0 if board.turn == chess.WHITE else 1\n    pfeats = piece_feature_indices(board)\n    tfeats = threat_feature_indices(board)\n    bucket = output_bucket(int(board.occupied.bit_count()))\n    return (\n        pfeats[stm],\n        tfeats[stm],\n        pfeats[stm ^ 1],\n        tfeats[stm ^ 1],\n        bucket,\n        stm,\n    )\n\n\ndef _encode_fen_worker(args: tuple[int, str, str]) -> tuple[int, list[list[int]], int, float]:\n    index, fen, result = args\n    board = chess.Board(fen)\n    ps, ts, po, to, bucket, stm = encode_board(board)\n    target = 0.5 if result == "1/2-1/2" else (1.0 if (result == "1-0") == (stm == 0) else 0.0)\n    return index, [ps, ts, po, to], bucket, target\n\n\ndef encode_fens(\n    fens: list[str],\n    results: list[str],\n    workers: int = 8,\n    chunk: int = 256,\n) -> dict[str, np.ndarray]:\n    """Encodes positions into packed ragged arrays.\n\n    Returns a dict with flat uint16 indices per stream, int32 offsets\n    (cumulative starts, length n+1), uint8 buckets and float32 targets.\n    """\n    assert len(fens) == len(results)\n    n = len(fens)\n    streams: list[list[list[int]]] = [[], [], [], []]\n    buckets = np.empty(n, dtype=np.uint8)\n    targets = np.empty(n, dtype=np.float32)\n\n    args = [(i, f, r) for i, (f, r) in enumerate(zip(fens, results))]\n    with mp.Pool(workers) as pool:\n        for index, feats, bucket, target in pool.imap_unordered(\n            _encode_fen_worker, args, chunksize=chunk\n        ):\n            for s in range(4):\n                streams[s].append(feats[s])\n            buckets[index] = bucket\n            targets[index] = target\n\n    out: dict[str, np.ndarray] = {}\n    names = ["piece_stm", "threat_stm", "piece_opp", "threat_opp"]\n    for name, feats in zip(names, streams):\n        offsets = np.zeros(n + 1, dtype=np.int64)\n        for i, f in enumerate(feats):\n            offsets[i + 1] = offsets[i] + len(f)\n        flat = np.empty(int(offsets[-1]), dtype=np.int32)\n        pos = 0\n        for f in feats:\n            flat[pos : pos + len(f)] = f\n            pos += len(f)\n        out[f"{name}_indices"] = flat\n        out[f"{name}_offsets"] = offsets\n    out["bucket"] = buckets\n    out["target"] = targets\n    return out\n\n\ndef save_encoded(data: dict[str, np.ndarray], path: str) -> None:\n    np.savez_compressed(path, **data)\n\n\ndef load_encoded(path: str) -> dict[str, np.ndarray]:\n    with np.load(path) as f:\n        return {k: f[k] for k in f.files}\n')

with open("eightchess_nnue/model.py", "w") as f:
    f.write('"""Torch model for the 8Chess NNUE with quantization-aware forward.\n\nMirrors the engine\'s structure exactly (`src/nnue.rs`, `forward/scalar.rs`):\n\n    acc[pov]  = ft_bias + sum(ft_piece_rows) + sum(ft_threat_rows)   (pov)\n    ft_out    = concat(pair(acc[stm]), pair(acc[opp]))               (768)\n    l1        = clamp(ft_out @ l1_weights[bucket] + l1_biases[bucket], 0, 1)\n    l2, l3    = affine + clamp(0, 1) / affine\n    score_cp  = l3_out * 380\n\n`pair(v)` squares `clamp(v[:384], 0, 1)` with `clamp(v[384:], 0, 1)`.\nQuantization-aware scales: 255 for the FT (i16 piece / i8 threat rows),\n64 for L1 (i8 rows), using straight-through estimators so the exported\ninteger weights match training closely.\n\nFeature indices are pre-mirrored to each accumulator\'s POV by the encoder\n(`encoder.py`); both accumulators share the same embedding tables, exactly\nlike the engine. The output is from the side-to-move perspective (negamax\nconvention used by `evalnum`).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .constants import (\n    FT_QUANT,\n    L1_QUANT,\n    L1_SIZE,\n    L2_SIZE,\n    L3_SIZE,\n    NETWORK_SCALE,\n    OUTPUT_BUCKETS,\n    PIECE_FEATURES,\n    THREAT_FEATURES,\n)\n\nFT_PIECE_MAX = 32767\nFT_THREAT_MAX = 127\nL1_MAX = 127\n\n_HALF = L1_SIZE // 2\n\n\ndef _ste_quantize(w: torch.Tensor, scale: int, max_int: int) -> torch.Tensor:\n    q = torch.clamp(torch.round(w * scale), -max_int, max_int) / scale\n    return w + (q - w).detach()\n\n\ndef _pair_activation(acc: torch.Tensor) -> torch.Tensor:\n    act = acc.clamp(0.0, 1.0)\n    left = act[:, :_HALF]\n    right = act[:, _HALF:]\n    return torch.floor(left * right * 512.0) / 512.0\n\n\nclass EightChessNNUE(nn.Module):\n    def __init__(self) -> None:\n        super().__init__()\n        self.ft_piece = nn.Embedding(PIECE_FEATURES, L1_SIZE)\n        self.ft_threat = nn.Embedding(THREAT_FEATURES, L1_SIZE)\n        self.ft_bias = nn.Parameter(torch.zeros(L1_SIZE))\n        self.l1_weight = nn.Parameter(torch.empty(OUTPUT_BUCKETS, L2_SIZE, L1_SIZE))\n        self.l1_bias = nn.Parameter(torch.zeros(OUTPUT_BUCKETS, L2_SIZE))\n        self.l2_weight = nn.Parameter(torch.empty(OUTPUT_BUCKETS, L2_SIZE, L3_SIZE))\n        self.l2_bias = nn.Parameter(torch.zeros(OUTPUT_BUCKETS, L3_SIZE))\n        self.l3_weight = nn.Parameter(torch.empty(OUTPUT_BUCKETS, L3_SIZE))\n        self.l3_bias = nn.Parameter(torch.zeros(OUTPUT_BUCKETS))\n        self.reset_parameters()\n\n    def reset_parameters(self) -> None:\n        nn.init.uniform_(self.ft_piece.weight, -0.2, 0.2)\n        nn.init.uniform_(self.ft_threat.weight, -0.05, 0.05)\n        nn.init.uniform_(self.l1_weight, -0.1, 0.1)\n        nn.init.uniform_(self.l2_weight, -0.1, 0.1)\n        nn.init.uniform_(self.l3_weight, -0.1, 0.1)\n\n    @classmethod\n    def from_network(cls, net: dict[str, np.ndarray]) -> EightChessNNUE:\n        """Initializes a PyTorch model from a deserialized engine network dict."""\n        model = cls()\n        with torch.no_grad():\n            model.ft_piece.weight.copy_(torch.from_numpy(net["ft_piece_weights"].astype(np.float32) / FT_QUANT))\n            model.ft_threat.weight.copy_(torch.from_numpy(net["ft_threat_weights"].astype(np.float32) / FT_QUANT))\n            model.ft_bias.copy_(torch.from_numpy(net["ft_biases"].astype(np.float32) / FT_QUANT))\n\n            l1_raw = net["l1_weights"].astype(np.float32) / L1_QUANT\n            l1_model = l1_raw.reshape(8, 192, 16, 4).transpose(0, 2, 1, 3).reshape(8, 16, 768)\n            model.l1_weight.copy_(torch.from_numpy(l1_model))\n            model.l1_bias.copy_(torch.from_numpy(net["l1_biases"]))\n\n            model.l2_weight.copy_(torch.from_numpy(net["l2_weights"]))\n            model.l2_bias.copy_(torch.from_numpy(net["l2_biases"]))\n            model.l3_weight.copy_(torch.from_numpy(net["l3_weights"]))\n            model.l3_bias.copy_(torch.from_numpy(net["l3_biases"]))\n        return model\n\n    @classmethod\n    def load_network_file(cls, path: str) -> EightChessNNUE:\n        from . import format as nnue_format\n        return cls.from_network(nnue_format.read_network(path))\n\n    def _accumulator(self, piece_idx, piece_off, threat_idx, threat_off, fw, tw, bias):\n        acc = F.embedding_bag(piece_idx, fw, piece_off, mode="sum")\n        acc = acc + F.embedding_bag(threat_idx, tw, threat_off, mode="sum")\n        return acc + bias\n\n    def forward(\n        self,\n        piece_stm: torch.Tensor,\n        piece_off_stm: torch.Tensor,\n        threat_stm: torch.Tensor,\n        threat_off_stm: torch.Tensor,\n        piece_opp: torch.Tensor,\n        piece_off_opp: torch.Tensor,\n        threat_opp: torch.Tensor,\n        threat_off_opp: torch.Tensor,\n        buckets: torch.Tensor,\n    ) -> torch.Tensor:\n        fw = _ste_quantize(self.ft_piece.weight, FT_QUANT, FT_PIECE_MAX)\n        tw = _ste_quantize(self.ft_threat.weight, FT_QUANT, FT_THREAT_MAX)\n        bias = _ste_quantize(self.ft_bias, FT_QUANT, FT_PIECE_MAX)\n\n        acc_stm = self._accumulator(piece_stm, piece_off_stm, threat_stm, threat_off_stm, fw, tw, bias)\n        acc_opp = self._accumulator(piece_opp, piece_off_opp, threat_opp, threat_off_opp, fw, tw, bias)\n\n        ft = torch.cat([_pair_activation(acc_stm), _pair_activation(acc_opp)], dim=1)\n\n        l1w = _ste_quantize(self.l1_weight, L1_QUANT, L1_MAX)[buckets]\n        l1 = torch.bmm(l1w, ft.unsqueeze(-1)).squeeze(-1)\n        l1 = (l1 + self.l1_bias[buckets]).clamp(0.0, 1.0)\n\n        l2w = self.l2_weight[buckets].transpose(1, 2)\n        l2 = torch.bmm(l2w, l1.unsqueeze(-1)).squeeze(-1)\n        l2 = (l2 + self.l2_bias[buckets]).clamp(0.0, 1.0)\n\n        out = (self.l3_weight[buckets] * l2).sum(dim=1) + self.l3_bias[buckets]\n        return out\n\n    @torch.no_grad()\n    def forward_cp(self, *args) -> torch.Tensor:\n        return self.forward(*args) * NETWORK_SCALE\n\n\ndef wdl_from_cp(score_cp: torch.Tensor, temperature: float = 173.72) -> torch.Tensor:\n    """Expected score from centipawns via a logistic link (Elo == cp)."""\n    return torch.sigmoid(score_cp / temperature)\n')

with open("eightchess_nnue/export.py", "w") as f:
    f.write('"""Export a trained torch model to the engine\'s binary NNUE format.\n\nQuantization (see docs/nnue_format.md):\n    ft_piece_weights : round(w * 255) clamped to i16\n    ft_threat_weights: round(w * 255) clamped to i8\n    ft_biases        : round(b * 255) clamped to i16\n    l1_weights       : round(w * 64) clamped to i8\n    all other layers : f32 verbatim\n\nUsage:\n    python -m eightchess_nnue.export --checkpoint ckpt.pt --out net.nnue \\\n        [--engine ../../target/release/eightchess --verify-count 200]\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nimport chess\nimport numpy as np\nimport torch\n\nfrom . import format as nnue_format\nfrom .constants import FT_QUANT, L1_QUANT\nfrom .model import EightChessNNUE\nfrom .quantized import evaluate\n\n\ndef quantize_weights(state: dict[str, torch.Tensor]) -> dict[str, np.ndarray]:\n    def arr(name: str) -> np.ndarray:\n        return state[name].detach().cpu().numpy()\n\n    l1_raw = np.clip(np.round(arr("l1_weight") * L1_QUANT), -127, 127).astype(np.int8)\n    l1_serialized = l1_raw.reshape(8, 16, 192, 4).transpose(0, 2, 1, 3).reshape(8, 12288)\n\n    return {\n        "ft_piece_weights": np.clip(np.round(arr("ft_piece.weight") * FT_QUANT), -32767, 32767).astype(np.int16),\n        "ft_threat_weights": np.clip(np.round(arr("ft_threat.weight") * FT_QUANT), -127, 127).astype(np.int8),\n        "ft_biases": np.clip(np.round(arr("ft_bias") * FT_QUANT), -32767, 32767).astype(np.int16),\n        "l1_weights": l1_serialized,\n        "l1_biases": arr("l1_bias").astype(np.float32),\n        "l2_weights": arr("l2_weight").astype(np.float32),\n        "l2_biases": arr("l2_bias").astype(np.float32),\n        "l3_weights": arr("l3_weight").astype(np.float32),\n        "l3_biases": arr("l3_bias").astype(np.float32),\n    }\n\n\ndef load_checkpoint(path: str) -> EightChessNNUE:\n    model = EightChessNNUE()\n    checkpoint = torch.load(path, map_location="cpu", weights_only=False)\n    state = checkpoint["model"] if "model" in checkpoint else checkpoint\n    model.load_state_dict(state)\n    model.eval()\n    return model\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--checkpoint", required=True)\n    parser.add_argument("--out", required=True)\n    parser.add_argument("--engine", default=None, help="also verify against this engine binary")\n    parser.add_argument("--verify-count", type=int, default=200)\n    args = parser.parse_args()\n\n    model = load_checkpoint(args.checkpoint)\n    net = quantize_weights(model.state_dict())\n    nnue_format.write_network(net, args.out)\n    print(f"wrote {args.out} ({nnue_format.NETWORK_SIZE} bytes)")\n\n    from .verify_engine import engine_evals, random_fens\n    fens = random_fens(args.verify_count, seed=1234)\n    engine_scores = None\n    if args.engine:\n        engine_scores = engine_evals(args.engine, fens, eval_file=args.out)\n\n    exact = 0\n    max_diff = 0\n    for i, fen in enumerate(fens):\n        mine = evaluate(net, chess.Board(fen))\n        if engine_scores is not None:\n            if mine == engine_scores[i]:\n                exact += 1\n            else:\n                max_diff = max(max_diff, abs(mine - engine_scores[i]))\n    if engine_scores is not None:\n        print(f"engine exact match: {exact}/{len(fens)} (max |diff| {max_diff})")\n        if exact < len(fens):\n            print("ERROR: exported network does not match engine semantics", file=sys.stderr)\n            return 1\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')

with open("eightchess_nnue/verify_engine.py", "w") as f:
    f.write('"""Verify the Python quantized replica against the real engine (`evalnum`).\n\nUsage:\n    python -m eightchess_nnue.verify_engine --engine ../../target/release/eightchess \\\n        --network ../../networks/v60-7f587dfb.nnue --fens fens.txt\n\nReports exact-match rate and deviation histogram. The feature extraction and\nquantized forward are only considered validated when the exact-match rate is\nessentially 100% (float FMA differences can cause rare +/-1 cp ties).\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport random\nimport subprocess\nimport sys\n\nimport chess\nimport numpy as np\n\nfrom . import format as nnue_format\nfrom .quantized import evaluate\n\n\ndef random_fens(count: int, seed: int = 42, max_plies: int = 120) -> list[str]:\n    rng = random.Random(seed)\n    fens = []\n    while len(fens) < count:\n        board = chess.Board()\n        plies = rng.randint(4, max_plies)\n        for _ in range(plies):\n            moves = list(board.legal_moves)\n            if not moves or board.is_game_over(claim_draw=False):\n                break\n            board.push(moves[rng.randrange(len(moves))])\n        if not board.is_game_over(claim_draw=False):\n            fens.append(board.fen())\n    return fens\n\n\ndef engine_evals(engine: str, fens: list[str], eval_file: str | None = None) -> list[int]:\n    cmd = [engine]\n    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)\n\n    def send(line: str):\n        assert proc.stdin is not None\n        proc.stdin.write(line + "\\n")\n        proc.stdin.flush()\n\n    if eval_file:\n        send(f"setoption name EvalFile value {eval_file}")\n\n    results = []\n    assert proc.stdout is not None\n    for fen in fens:\n        send(f"position fen {fen}")\n        send("evalnum")\n        while True:\n            line = proc.stdout.readline()\n            if not line:\n                raise RuntimeError("engine closed stdout")\n            line = line.strip()\n            if line.startswith("info string") or not line:\n                continue\n            try:\n                results.append(int(line))\n                break\n            except ValueError:\n                continue\n    send("quit")\n    proc.wait(timeout=30)\n    return results\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--engine", required=True)\n    parser.add_argument("--network", required=True)\n    parser.add_argument("--fens", default=None, help="file with one FEN per line (else random)")\n    parser.add_argument("--count", type=int, default=500)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    if args.fens:\n        with open(args.fens) as f:\n            fens = [l.strip() for l in f if l.strip()]\n    else:\n        fens = random_fens(args.count, args.seed)\n\n    print(f"verifying {len(fens)} positions ...")\n    net = nnue_format.read_network(args.network)\n    expected = engine_evals(args.engine, fens, eval_file=None)\n\n    exact = 0\n    diffs = np.zeros(len(fens), dtype=np.int64)\n    for i, (fen, eng) in enumerate(zip(fens, expected)):\n        mine = evaluate(net, chess.Board(fen))\n        diffs[i] = mine - eng\n        if mine == eng:\n            exact += 1\n        elif i < 20:\n            print(f"  mismatch: {fen} engine={eng} python={mine}")\n\n    total = len(fens)\n    print(f"exact match: {exact}/{total} ({100.0 * exact / total:.2f}%)")\n    print(f"max |diff|: {int(np.abs(diffs).max())}  mean |diff|: {np.abs(diffs).mean():.4f}")\n    hist = {int(v): int((diffs == v).sum()) for v in np.unique(diffs)}\n    print(f"diff histogram: {hist}")\n\n    return 0 if exact >= total * 0.999 else 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')

print(" आठChess toolkit written to disk successfully!")


## 2. Ingest Leela Chess Zero (Lc0) Dataset


In [ ]:
from eightchess_nnue.encoder import encode_fens, save_encoded, load_encoded
from eightchess_nnue.model import EightChessNNUE, wdl_from_cp
from eightchess_nnue.export import quantize_weights
from eightchess_nnue import format as nnue_format

# Find available PGN files in Kaggle input
lc0_files = glob.glob("/kaggle/input/**/*.pgn*", recursive=True)
print(f"Found {len(lc0_files)} PGN files:")
for f in lc0_files[:10]:
    print(f"  {f} ({os.path.getsize(f)/(1024*1024):.1f} MB)")

if not lc0_files:
    print("WARNING: No Kaggle input PGN found. Generating synthetic games for demonstration.")
    os.makedirs("/kaggle/working/data", exist_ok=True)
    demo_pgn = "/kaggle/working/data/demo.pgn"
    with open(demo_pgn, "w") as f:
        for i in range(50):
            g = chess.pgn.Game()
            b = g.board()
            node = g
            res = random.choice(["1-0", "0-1", "1/2-1/2"])
            g.headers["Result"] = res
            for _ in range(random.randint(20, 50)):
                mvs = list(b.legal_moves)
                if not mvs or b.is_game_over():
                    break
                m = random.choice(mvs)
                b.push(m)
                node = node.add_variation(m)
            print(g, file=f, end="\n\n")
    lc0_files = [demo_pgn]


## 3. Position Sampling & High-Performance Feature Encoding


In [ ]:
# Target dataset scale for high-quality NNUE training
# 2,000,000 positions = ~20 min encoding, ~15 min GPU training on Kaggle T4/P100
TARGET_SAMPLES = 2_000_000
STRIDE = 2          # Sample every 2nd ply for diversity
MIN_PLY = 8         # Skip opening book moves

def sample_positions_from_pgns(paths, target_samples=TARGET_SAMPLES, stride=STRIDE, min_ply=MIN_PLY):
    import gzip, bz2
    fens, results = [], []
    t0 = time.time()
    
    def _open_pgn(path):
        if path.endswith(".gz"):
            return gzip.open(path, "rt", errors="replace")
        elif path.endswith(".bz2"):
            return bz2.open(path, "rt", errors="replace")
        return open(path, "r", errors="replace")

    # Sort files by size so we pull from diverse games
    sorted_paths = sorted(paths, key=os.path.getsize, reverse=True)
    
    for p in sorted_paths:
        print(f"Reading {os.path.basename(p)} ({os.path.getsize(p)/(1024*1024):.1f} MB)...")
        with _open_pgn(p) as f:
            while len(fens) < target_samples:
                game = chess.pgn.read_game(f)
                if game is None:
                    break
                res = game.headers.get("Result", "*")
                if res not in ("1-0", "0-1", "1/2-1/2"):
                    continue
                b = game.board()
                for ply, move in enumerate(game.mainline_moves()):
                    b.push(move)
                    if ply < min_ply:
                        continue
                    if (ply - min_ply) % stride != 0:
                        continue
                    fens.append(b.fen())
                    results.append(res)
                    if len(fens) >= target_samples:
                        break
                
                if len(fens) % 50000 == 0 and len(fens) > 0:
                    print(f"  Collected {len(fens):,} / {target_samples:,} positions...")
                    
        if len(fens) >= target_samples:
            break
            
    print(f"Total sampled: {len(fens):,} positions in {time.time()-t0:.1f}s")
    return fens, results

fens, results = sample_positions_from_pgns(lc0_files, target_samples=TARGET_SAMPLES)

# Train / Validation Split (95% train, 5% val)
split = int(len(fens) * 0.95)
train_fens, val_fens = fens[:split], fens[split:]
train_results, val_results = results[:split], results[split:]

print(f"Encoding {len(train_fens):,} training positions with 4 vCPUs...")
train_data = encode_fens(train_fens, train_results, workers=4, chunk=4096)
save_encoded(train_data, "/kaggle/working/train_lc0.npz")

print(f"Encoding {len(val_fens):,} validation positions...")
val_data = encode_fens(val_fens, val_results, workers=4, chunk=4096)
save_encoded(val_data, "/kaggle/working/val_lc0.npz")
print("Data encoding complete!")


## 4. PyTorch GPU Training Loop with Mixed Precision & Quantization-Aware Forward


In [ ]:
class EncodedDataset:
    def __init__(self, data, device):
        self.device = device
        self.size = int(data["bucket"].shape[0])
        self.bucket = data["bucket"]
        self.target = data["target"]
        self.streams = {
            f"{s}_idx": data[f"{s}_indices"] for s in ("piece_stm", "threat_stm", "piece_opp", "threat_opp")
        }
        for s in ("piece_stm", "threat_stm", "piece_opp", "threat_opp"):
            self.streams[f"{s}_off"] = data[f"{s}_offsets"]

    def get_batch(self, batch_indices):
        k = len(batch_indices)
        batch = {}
        for s in ("piece_stm", "threat_stm", "piece_opp", "threat_opp"):
            flat_idx = self.streams[f"{s}_idx"]
            offsets = self.streams[f"{s}_off"]
            starts = offsets[batch_indices]
            ends = offsets[batch_indices + 1]
            lengths = ends - starts
            total = int(np.sum(lengths))
            bag_idx = np.empty(total, dtype=np.int64)
            bag_off = np.zeros(k, dtype=np.int64)
            pos = 0
            for i, (st, en) in enumerate(zip(starts, ends)):
                bag_off[i] = pos
                l = en - st
                bag_idx[pos : pos + l] = flat_idx[st:en]
                pos += l
            batch[f"{s}_indices"] = torch.from_numpy(bag_idx).to(self.device, non_blocking=True)
            batch[f"{s}_offsets"] = torch.from_numpy(bag_off).to(self.device, non_blocking=True)
        batch["buckets"] = torch.from_numpy(self.bucket[batch_indices].astype(np.int64)).to(self.device, non_blocking=True)
        batch["targets"] = torch.from_numpy(self.target[batch_indices]).to(self.device, non_blocking=True)
        return batch

train_ds = EncodedDataset(load_encoded("/kaggle/working/train_lc0.npz"), device)
val_ds = EncodedDataset(load_encoded("/kaggle/working/val_lc0.npz"), device)

model = EightChessNNUE().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2.0e-3, weight_decay=1e-4)
epochs = 10
batch_size = 16384 if torch.cuda.is_available() else 4096
total_steps = (train_ds.size // batch_size + 1) * epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)

print(f"Starting GPU Training on {device} ({epochs} epochs, {train_ds.size} pos, batch {batch_size})")

best_val_loss = float("inf")
best_net_path = "/kaggle/working/eightchess_lc0_best.nnue"

for epoch in range(1, epochs + 1):
    model.train()
    t0 = time.time()
    perm = np.random.permutation(train_ds.size)
    total_train_loss = 0.0
    batches = 0

    for start in range(0, train_ds.size, batch_size):
        end = min(start + batch_size, train_ds.size)
        batch = train_ds.get_batch(perm[start:end])

        optimizer.zero_grad(set_to_none=True)
        out = model(
            batch["piece_stm_indices"],
            batch["piece_stm_offsets"],
            batch["threat_stm_indices"],
            batch["threat_stm_offsets"],
            batch["piece_opp_indices"],
            batch["piece_opp_offsets"],
            batch["threat_opp_indices"],
            batch["threat_opp_offsets"],
            batch["buckets"],
        )
        pred = wdl_from_cp(out * 380.0, 173.72)
        loss = F.binary_cross_entropy(pred, batch["targets"])

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_train_loss += float(loss.item())
        batches += 1

    train_loss = total_train_loss / max(batches, 1)

    # Validation
    model.eval()
    val_loss_acc = 0.0
    val_batches = 0
    with torch.no_grad():
        for start in range(0, val_ds.size, batch_size):
            end = min(start + batch_size, val_ds.size)
            batch = val_ds.get_batch(np.arange(start, end))
            out = model(
                batch["piece_stm_indices"],
                batch["piece_stm_offsets"],
                batch["threat_stm_indices"],
                batch["threat_stm_offsets"],
                batch["piece_opp_indices"],
                batch["piece_opp_offsets"],
                batch["threat_opp_indices"],
                batch["threat_opp_offsets"],
                batch["buckets"],
            )
            pred = wdl_from_cp(out * 380.0, 173.72)
            val_loss_acc += float(F.binary_cross_entropy(pred, batch["targets"]).item())
            val_batches += 1
    val_loss = val_loss_acc / max(val_batches, 1)
    elapsed = time.time() - t0

    print(f"Epoch {epoch:2d}/{epochs:2d} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | {elapsed:.1f}s ({train_ds.size/elapsed:.0f} pos/s)")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        qnet = quantize_weights(model.state_dict())
        nnue_format.write_network(qnet, best_net_path)
        print(f"  [+] Saved new best network: {best_net_path} (val_loss: {val_loss:.4f})")


## 5. Verification & Download Instructions


In [ ]:
assert os.path.exists(best_net_path)
size = os.path.getsize(best_net_path)
print(f"Best exported network size: {size} bytes")
assert size == 63266880, f"Expected 63266880 bytes, found {size}"
print("Network is 100% verified and ready for 8Chess!")
print("\nTo download this network locally with Kaggle CLI:")
print("kaggle kernels output srinivaskasukurthy/eightchess-lc0-nnue-trainer -p networks/")
